# 1.&nbsp;UnzippingData

THE FOLLOWING SCRIPTS ARE GENERATED BY CLAUDE AI 3.7

### RAFDB AND FERPLUS

In [ ]:
# imports
import os
import zipfile
import time
import shutil

# Define paths - updated for JupyterHub environment
# Assuming your zip files are uploaded to your home directory
HOME_PATH = '/home/jovyan/gaave'  
ZIP_PATH = os.path.join(HOME_PATH, 'data_zips')  # Where you store your zip files
LOCAL_BASE_PATH = os.path.join(HOME_PATH, 'data')  # Local destination for unzipped files

# Create local directory
os.makedirs(LOCAL_BASE_PATH, exist_ok=True)

# Function to unzip a file
def unzip_file(zip_path, extract_to):
    """Unzip a file from zip_path to extract_to"""
    if not os.path.exists(zip_path):
        print(f"Zip file not found: {zip_path}")
        return False

    os.makedirs(extract_to, exist_ok=True)

    print(f"Unzipping {zip_path} to {extract_to}...")
    start_time = time.time()

    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)

        elapsed = time.time() - start_time
        print(f"Successfully unzipped in {elapsed:.2f} seconds")
        return True
    except Exception as e:
        print(f"Error unzipping {zip_path}: {e}")
        return False

# Function to organize RAF-DB files - unchanged
def organize_rafdb(base_dir):
    """
    Organize RAF-DB files by placing 'aligned' folder and label file into 'rafdb' directory
    """
    # Create rafdb directory if it doesn't exist
    rafdb_dir = os.path.join(base_dir, 'rafdb')
    os.makedirs(rafdb_dir, exist_ok=True)

    # Look for 'aligned' folder
    aligned_dir = os.path.join(base_dir, 'aligned')
    if os.path.exists(aligned_dir):
        # Move aligned folder to rafdb directory
        dest_aligned_dir = os.path.join(rafdb_dir, 'aligned')
        if not os.path.exists(dest_aligned_dir):
            print(f"Moving 'aligned' folder to {dest_aligned_dir}")
            shutil.move(aligned_dir, dest_aligned_dir)
        else:
            print(f"'{dest_aligned_dir}' already exists")
    else:
        # Check if it's already in the right place
        if os.path.exists(os.path.join(rafdb_dir, 'aligned')):
            print("'aligned' folder already in correct location")
        else:
            # Look for it elsewhere
            for root, dirs, _ in os.walk(base_dir):
                if 'aligned' in dirs:
                    src_aligned = os.path.join(root, 'aligned')
                    dest_aligned = os.path.join(rafdb_dir, 'aligned')
                    print(f"Found 'aligned' folder at {src_aligned}, moving to {dest_aligned}")
                    shutil.move(src_aligned, dest_aligned)
                    break
            else:
                print("Warning: 'aligned' folder not found")

    # Look for label file (both common names)
    label_files = ['list_partition_label.txt', 'list_patition_label.txt']
    found_label = False

    for label_file in label_files:
        src_label = os.path.join(base_dir, label_file)
        if os.path.exists(src_label):
            # Move label file to rafdb directory
            dest_label = os.path.join(rafdb_dir, label_file)
            if not os.path.exists(dest_label):
                print(f"Moving label file to {dest_label}")
                shutil.move(src_label, dest_label)
                found_label = True
            else:
                print(f"'{dest_label}' already exists")
                found_label = True

    if not found_label:
        # Check if it's already in the right place
        for label_file in label_files:
            if os.path.exists(os.path.join(rafdb_dir, label_file)):
                print("Label file already in correct location")
                found_label = True
                break

        if not found_label:
            # Look for it elsewhere
            for label_file in label_files:
                for root, _, files in os.walk(base_dir):
                    if label_file in files:
                        src_label = os.path.join(root, label_file)
                        dest_label = os.path.join(rafdb_dir, label_file)
                        print(f"Found label file at {src_label}, moving to {dest_label}")
                        shutil.move(src_label, dest_label)
                        found_label = True
                        break
                if found_label:
                    break
            else:
                print("Warning: Label file not found")

# Datasets to unzip - update paths to point to your JupyterHub directory
datasets = {
    'ferplus': os.path.join(ZIP_PATH, 'ferplus.zip'),
    'rafdb': os.path.join(ZIP_PATH, 'rafdb.zip')
}

# Unzip each dataset
for dataset_name, zip_path in datasets.items():
    print(f"\n=== Unzipping {dataset_name} dataset ===")
    success = unzip_file(zip_path, LOCAL_BASE_PATH)

    if success:
        dataset_dir = os.path.join(LOCAL_BASE_PATH, dataset_name)
        # Check if the directory was created by the unzip process
        if not os.path.exists(dataset_dir):
            # Try to find the actual extracted directory
            print(f"Looking for extracted {dataset_name} directory...")
            extracted = False
            for item in os.listdir(LOCAL_BASE_PATH):
                item_path = os.path.join(LOCAL_BASE_PATH, item)
                if os.path.isdir(item_path) and dataset_name.lower() in item.lower():
                    print(f"Found directory that might contain {dataset_name} data: {item}")
                    # Optionally rename for consistency
                    new_path = os.path.join(LOCAL_BASE_PATH, dataset_name)
                    if item_path != new_path:
                        print(f"Renaming {item_path} to {new_path}")
                        os.rename(item_path, new_path)
                    extracted = True
                    break

            if not extracted:
                print(f"Warning: Could not find extracted {dataset_name} directory")

# Special handling for RAF-DB
print("\n=== Organizing RAF-DB dataset ===")
organize_rafdb(LOCAL_BASE_PATH)

# Count files in extracted directories
def count_files(directory):
    """Count the number of files in a directory (recursive)"""
    if not os.path.exists(directory):
        return 0

    count = 0
    for root, dirs, files in os.walk(directory):
        count += len(files)
    return count

# Verify files were extracted
print("\n=== Verification ===")
for dataset_name in datasets.keys():
    dataset_dir = os.path.join(LOCAL_BASE_PATH, dataset_name)
    file_count = count_files(dataset_dir)
    print(f"{dataset_name}: {file_count} files extracted")

# Special verification for RAF-DB
rafdb_dir = os.path.join(LOCAL_BASE_PATH, 'rafdb')
if os.path.exists(os.path.join(rafdb_dir, 'aligned')):
    print("✓ RAF-DB 'aligned' folder is in the correct location")
else:
    print("✗ RAF-DB 'aligned' folder not found in the correct location")

label_found = False
for label_file in ['list_partition_label.txt', 'list_patition_label.txt']:
    if os.path.exists(os.path.join(rafdb_dir, label_file)):
        print(f"✓ RAF-DB label file '{label_file}' is in the correct location")
        label_found = True
        break
if not label_found:
    print("✗ RAF-DB label file not found in the correct location")

print("\nDone! Your files have been extracted to JupyterHub storage.")
print(f"Update your code to use this data directory: args.data_dir = '{LOCAL_BASE_PATH}'")

### AFFWILD2

In [ ]:
# imports
import os
import zipfile
import time
import shutil

# Define paths - updated for JupyterHub environment
# Assuming your zip files are uploaded to your home directory
HOME_PATH = '/home/jovyan/gaave'  
ZIP_PATH = os.path.join(HOME_PATH, 'data_zips')  # Where you store your zip files
LOCAL_BASE_PATH = os.path.join(HOME_PATH, 'data')  # Local destination for unzipped files

# Create local directory
os.makedirs(LOCAL_BASE_PATH, exist_ok=True)

# Function to unzip a file with progress reporting
def unzip_file(zip_path, extract_to):
    """Unzip a file from zip_path to extract_to with progress reporting"""
    if not os.path.exists(zip_path):
        print(f"Zip file not found: {zip_path}")
        return False

    os.makedirs(extract_to, exist_ok=True)

    print(f"Unzipping {zip_path} to {extract_to}...")
    start_time = time.time()

    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            # Get total number of files for progress reporting
            total_files = len(zip_ref.namelist())
            print(f"Total files to extract: {total_files}")

            # Extract in batches with progress reporting
            for i, file in enumerate(zip_ref.namelist()):
                if i % 100000 == 0 and i > 0:
                    elapsed = time.time() - start_time
                    print(f"Progress: {i}/{total_files} files ({(i/total_files)*100:.1f}%) - {elapsed:.1f} seconds elapsed")
                zip_ref.extract(file, extract_to)

        elapsed = time.time() - start_time
        print(f"Successfully unzipped in {elapsed:.2f} seconds")
        return True
    except Exception as e:
        print(f"Error unzipping {zip_path}: {e}")
        return False

# Function to organize AffWild2 content
def organize_affwild2(base_dir):
    """
    Organize AffWild2 content by moving relevant folders into a single affwild2 directory
    """
    # Create affwild2 directory if it doesn't exist
    affwild2_dir = os.path.join(base_dir, 'affwild2')
    os.makedirs(affwild2_dir, exist_ok=True)

    # Look for any potential extracted folders from AffWild2 zips
    for root, dirs, _ in os.walk(base_dir):
        for dir_name in dirs:
            # Skip the __MACOSX folders, but process cropped_aligned folders
            if dir_name.startswith('__MACOSX'):
                continue

            if 'cropped_aligned' in dir_name.lower():
                src_dir = os.path.join(root, dir_name)

                # If this directory contains video folders, move each to the affwild2 dir
                for item in os.listdir(src_dir):
                    item_path = os.path.join(src_dir, item)
                    if os.path.isdir(item_path):
                        dest_path = os.path.join(affwild2_dir, item)
                        if not os.path.exists(dest_path):
                            print(f"Moving {item} to {dest_path}")
                            shutil.move(item_path, dest_path)
                        else:
                            print(f"{dest_path} already exists, skipping")

                # After moving all content, remove the empty source directory
                if len(os.listdir(src_dir)) == 0:
                    print(f"Removing empty directory: {src_dir}")
                    shutil.rmtree(src_dir)

# Function to organize AffWild2 annotations
def organize_affwild2_annotations(base_dir):
    """
    Organize AffWild2 annotations by setting up the proper directories
    """
    # Create annotations directory structure
    affwild2_dir = os.path.join(base_dir, 'affwild2')
    annotations_dir = os.path.join(affwild2_dir, 'annotations')
    os.makedirs(annotations_dir, exist_ok=True)

    # Create subdirectories for each challenge
    expr_dir = os.path.join(annotations_dir, 'EXPR')
    au_dir = os.path.join(annotations_dir, 'AU')
    va_dir = os.path.join(annotations_dir, 'VA')

    os.makedirs(expr_dir, exist_ok=True)
    os.makedirs(au_dir, exist_ok=True)
    os.makedirs(va_dir, exist_ok=True)

    # Find the extracted 6th ABAW Annotations folder
    abaw_annotations_path = None

    for root, dirs, _ in os.walk(base_dir):
        for dir_name in dirs:
            if '6th ABAW Annotations' in dir_name:
                abaw_annotations_path = os.path.join(root, dir_name)
                break
        if abaw_annotations_path:
            break

    if not abaw_annotations_path:
        print("Warning: Could not find '6th ABAW Annotations' folder")
        return

    print(f"Found annotations at: {abaw_annotations_path}")

    # Process EXPR annotations (Expression Recognition Challenge)
    expr_src_dir = None
    for root, dirs, _ in os.walk(abaw_annotations_path):
        for dir_name in dirs:
            if 'EXPR_Recognition_Challenge' in dir_name:
                expr_src_dir = os.path.join(root, dir_name)
                break
        if expr_src_dir:
            break

    if expr_src_dir:
        print(f"Found EXPR annotations at: {expr_src_dir}")

        # Copy Train_Set to annotations/EXPR/train
        train_src = os.path.join(expr_src_dir, 'Train_Set')
        if os.path.exists(train_src):
            train_dest = os.path.join(expr_dir, 'train')
            if not os.path.exists(train_dest):
                print(f"Copying Train_Set to {train_dest}")
                shutil.copytree(train_src, train_dest)
            else:
                print(f"{train_dest} already exists, skipping")

        # Copy Validation_Set to annotations/EXPR/test
        val_src = os.path.join(expr_src_dir, 'Validation_Set')
        if os.path.exists(val_src):
            test_dest = os.path.join(expr_dir, 'test')
            if not os.path.exists(test_dest):
                print(f"Copying Validation_Set to {test_dest}")
                shutil.copytree(val_src, test_dest)
            else:
                print(f"{test_dest} already exists, skipping")
    else:
        print("Warning: Could not find EXPR_Recognition_Challenge folder")

    # For future: Add similar blocks for AU_Detection_Challenge and VA_Estimation_Challenge
    # Currently only implementing EXPR as in the original code

    print("Annotations organization complete")

# Count files in extracted directories
def count_files(directory):
    """Count the number of files in a directory (recursive)"""
    if not os.path.exists(directory):
        return 0

    count = 0
    for root, dirs, files in os.walk(directory):
        count += len(files)
    return count

# Datasets to unzip (AffWild2 data and annotations)
datasets = {
    'affwild2_batch1': os.path.join(ZIP_PATH, 'affwild2_batch1.zip'),
    'affwild2_batch2': os.path.join(ZIP_PATH, 'affwild2_batch2.zip'),
    'affwild2_annotations': os.path.join(ZIP_PATH, 'affwild2_annotations.zip')
}

# Unzip datasets
for dataset_name, zip_path in datasets.items():
    print(f"\n=== Unzipping {dataset_name} dataset ===")
    success = unzip_file(zip_path, LOCAL_BASE_PATH)
    
    if not success:
        print(f"Warning: Failed to unzip {dataset_name}")

# Organize AffWild2 files
print("\n=== Organizing AffWild2 dataset ===")
organize_affwild2(LOCAL_BASE_PATH)

# Organize AffWild2 annotations
print("\n=== Organizing AffWild2 annotations ===")
organize_affwild2_annotations(LOCAL_BASE_PATH)

# Verification
print("\n=== Verification ===")
print("Directory structure verification:")
affwild2_dir = os.path.join(LOCAL_BASE_PATH, 'affwild2')
if os.path.exists(affwild2_dir):
    # List top-level directories to verify structure
    print(f"AffWild2 directory exists at: {affwild2_dir}")
    items = os.listdir(affwild2_dir)
    dirs = [d for d in items if os.path.isdir(os.path.join(affwild2_dir, d))]
    print(f"Top-level directories in affwild2: {dirs}")

    # Check annotations
    annotations_dir = os.path.join(affwild2_dir, 'annotations')
    if os.path.exists(annotations_dir):
        print(f"Annotations directory exists at: {annotations_dir}")
        anno_dirs = [d for d in os.listdir(annotations_dir) if os.path.isdir(os.path.join(annotations_dir, d))]
        print(f"Challenge directories in annotations: {anno_dirs}")

        # Check EXPR annotations
        expr_dir = os.path.join(annotations_dir, 'EXPR')
        if os.path.exists(expr_dir):
            expr_dirs = [d for d in os.listdir(expr_dir) if os.path.isdir(os.path.join(expr_dir, d))]
            print(f"EXPR annotation sets: {expr_dirs}")

            # Count annotation files
            train_anno_count = count_files(os.path.join(expr_dir, 'train'))
            test_anno_count = count_files(os.path.join(expr_dir, 'test'))
            print(f"EXPR train annotations: {train_anno_count} files")
            print(f"EXPR test annotations: {test_anno_count} files")

    # Count video directories and frames
    video_dirs = [d for d in items if os.path.isdir(os.path.join(affwild2_dir, d)) and d != 'annotations']
    print(f"Found {len(video_dirs)} video directories")
    if len(video_dirs) > 0:
        print(f"Sample video directories: {video_dirs[:5]}")
else:
    print("Warning: AffWild2 directory not found")

# Cleanup any remaining temporary or empty directories
print("\n=== Cleaning up temporary directories ===")
for item in os.listdir(LOCAL_BASE_PATH):
    item_path = os.path.join(LOCAL_BASE_PATH, item)
    if os.path.isdir(item_path) and (
        '__MACOSX' in item or
        'cropped_aligned' in item.lower() or
        '6th ABAW Annotations' in item
    ):
        print(f"Removing temporary directory: {item_path}")
        shutil.rmtree(item_path)

print("\nDone! Your AffWild2 files have been extracted to JupyterHub storage.")
print(f"Update your code to use this data directory: args.data_dir = '{LOCAL_BASE_PATH}'")

### AffectNet7

In [ ]:
# imports
import os
import zipfile
import time
import shutil

# Define paths - updated for JupyterHub environment
HOME_PATH = '/home/jovyan/gaave'  
ZIP_PATH = os.path.join(HOME_PATH, 'data_zips')  # Where you store your zip files
LOCAL_BASE_PATH = os.path.join(HOME_PATH, 'data')  # Local destination for unzipped files

# Create local directory
os.makedirs(LOCAL_BASE_PATH, exist_ok=True)

# Function to unzip a file
def unzip_file(zip_path, extract_to):
    """Unzip a file from zip_path to extract_to"""
    if not os.path.exists(zip_path):
        print(f"Zip file not found: {zip_path}")
        return False

    os.makedirs(extract_to, exist_ok=True)

    print(f"Unzipping {zip_path} to {extract_to}...")
    start_time = time.time()

    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)

        elapsed = time.time() - start_time
        print(f"Successfully unzipped in {elapsed:.2f} seconds")
        return True
    except Exception as e:
        print(f"Error unzipping {zip_path}: {e}")
        return False

# Function to organize AffectNet-7 files
def organize_affectnet7(base_dir):
    """
    Organize AffectNet-7 files by placing train and valid folders and txt files into 'affectnet7' directory
    """
    # Create affectnet7 directory if it doesn't exist
    affectnet7_dir = os.path.join(base_dir, 'affectnet7')
    os.makedirs(affectnet7_dir, exist_ok=True)
    
    # Look for affectnet train and valid folders
    for folder_name in ['affectnet_train', 'affectnet_valid']:
        src_folder = os.path.join(base_dir, folder_name)
        if os.path.exists(src_folder):
            # Move folder to affectnet7 directory
            dest_folder = os.path.join(affectnet7_dir, folder_name)
            if not os.path.exists(dest_folder):
                print(f"Moving '{folder_name}' folder to {dest_folder}")
                shutil.move(src_folder, dest_folder)
            else:
                print(f"'{dest_folder}' already exists")
        else:
            # Check if it's already in the right place
            if os.path.exists(os.path.join(affectnet7_dir, folder_name)):
                print(f"'{folder_name}' folder already in correct location")
            else:
                # Look for it elsewhere
                for root, dirs, _ in os.walk(base_dir):
                    if folder_name in dirs:
                        src_folder = os.path.join(root, folder_name)
                        dest_folder = os.path.join(affectnet7_dir, folder_name)
                        print(f"Found '{folder_name}' folder at {src_folder}, moving to {dest_folder}")
                        shutil.move(src_folder, dest_folder)
                        break
                else:
                    print(f"Warning: '{folder_name}' folder not found")
    
    # Look for affectnet txt files - updated to include both train and test files
    txt_files = [
        'affectnet_train_according_to_emotiw_1.txt', 
        'affectnet_valid_according_to_emotiw_1.txt',
        'affectnet_train_test.txt',
        'affectnet_valid_test.txt'
    ]
    
    for txt_file in txt_files:
        src_txt = os.path.join(base_dir, txt_file)
        if os.path.exists(src_txt):
            # Move txt file to affectnet7 directory
            dest_txt = os.path.join(affectnet7_dir, txt_file)
            if not os.path.exists(dest_txt):
                print(f"Moving '{txt_file}' to {dest_txt}")
                shutil.move(src_txt, dest_txt)
            else:
                print(f"'{dest_txt}' already exists")
        else:
            # Check if it's already in the right place
            if os.path.exists(os.path.join(affectnet7_dir, txt_file)):
                print(f"'{txt_file}' already in correct location")
            else:
                # Look for it elsewhere
                for root, _, files in os.walk(base_dir):
                    if txt_file in files:
                        src_txt = os.path.join(root, txt_file)
                        dest_txt = os.path.join(affectnet7_dir, txt_file)
                        print(f"Found '{txt_file}' at {src_txt}, moving to {dest_txt}")
                        shutil.move(src_txt, dest_txt)
                        break
                else:
                    print(f"Warning: '{txt_file}' not found")

# Only AffectNet7 datasets 
datasets = {
    'affectnet_valid': os.path.join(ZIP_PATH, 'affectnet_valid.zip'),
    'affectnet_train': os.path.join(ZIP_PATH, 'affectnet_train.zip')
    
}

# Unzip each dataset
for dataset_name, zip_path in datasets.items():
    print(f"\n=== Unzipping {dataset_name} dataset ===")
    success = unzip_file(zip_path, LOCAL_BASE_PATH)

    if success:
        dataset_dir = os.path.join(LOCAL_BASE_PATH, dataset_name)
        # Check if the directory was created by the unzip process
        if not os.path.exists(dataset_dir):
            # Try to find the actual extracted directory
            print(f"Looking for extracted {dataset_name} directory...")
            extracted = False
            for item in os.listdir(LOCAL_BASE_PATH):
                item_path = os.path.join(LOCAL_BASE_PATH, item)
                if os.path.isdir(item_path) and dataset_name.lower() in item.lower():
                    print(f"Found directory that might contain {dataset_name} data: {item}")
                    # Optionally rename for consistency
                    new_path = os.path.join(LOCAL_BASE_PATH, dataset_name)
                    if item_path != new_path:
                        print(f"Renaming {item_path} to {new_path}")
                        os.rename(item_path, new_path)
                    extracted = True
                    break

            if not extracted:
                print(f"Warning: Could not find extracted {dataset_name} directory")

# Special handling for AffectNet-7
print("\n=== Organizing AffectNet-7 dataset ===")
organize_affectnet7(LOCAL_BASE_PATH)

# Count files in extracted directories
def count_files(directory):
    """Count the number of files in a directory (recursive)"""
    if not os.path.exists(directory):
        return 0

    count = 0
    for root, dirs, files in os.walk(directory):
        count += len(files)
    return count

# Verify files were extracted
print("\n=== Verification ===")

# Special verification for AffectNet-7
affectnet7_dir = os.path.join(LOCAL_BASE_PATH, 'affectnet7')
print(f"affectnet7: {count_files(affectnet7_dir)} files extracted")

# Verify AffectNet folders and files
for folder_name in ['affectnet_train', 'affectnet_valid']:
    if os.path.exists(os.path.join(affectnet7_dir, folder_name)):
        print(f"✓ AffectNet-7 '{folder_name}' folder is in the correct location")
    else:
        print(f"✗ AffectNet-7 '{folder_name}' folder not found in the correct location")

for txt_file in [
    'affectnet_train_according_to_emotiw_1.txt', 
    'affectnet_valid_according_to_emotiw_1.txt',
    'affectnet_train_test.txt',
    'affectnet_valid_test.txt'
]:
    if os.path.exists(os.path.join(affectnet7_dir, txt_file)):
        print(f"✓ AffectNet-7 txt file '{txt_file}' is in the correct location")
    else:
        print(f"✗ AffectNet-7 txt file '{txt_file}' not found in the correct location")

print("\nDone! AffectNet7 files have been extracted to JupyterHub storage.")
print(f"Update your code to use this data directory: args.data_dir = '{LOCAL_BASE_PATH}'")

# 2.&nbsp;Imports, Hyperparameters and Data Paths (args)

In [ ]:
# imports for main and GAAVE
import os
import time
import numpy as np
import argparse
from datetime import datetime
import sys

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import WeightedRandomSampler

import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights
from PIL import Image

import math
import random
from collections import Counter

In [ ]:
# arguments parser for hyperparameters
parser = argparse.ArgumentParser(
    prog='Emotion Recognition',
    description='ResNet18 for Emotion Recognition'
)

# hyperparameters
parser.add_argument('--epochs', type=int, default=200, help='num_training_epochs')
parser.add_argument('--batch_size', type=int, default=128, help='batch_size')
# regularisation hparameters
parser.add_argument('--learning_rate', type=float, default=1e-3, help='learning_rate')
parser.add_argument('--weight_decay', type=float, default=1e-5, help='weight_decay')
parser.add_argument('--momentum', type=float, default=0.9, help='momentum')
parser.add_argument('--label_smoothing', type=float, default=0.1, help='label smoothing factor')
# optimisation hparameters
parser.add_argument('--optimizer', type=str, default='ADAM', help='optimizer - choose either ADAM or SGD')
# dataset hparameters
parser.add_argument('--dataset', type=str, default='affectnet7', help='dataset name: rafdb, affectnet, ferplus, affwild2')

parser.add_argument('--subsampled', type=bool, default=False, help='use subsampled version of affwild2?')

parser.add_argument('--pretrained_path', type=str, default='/home/jovyan/gaave/resnet18_msceleb.pth', help='data directory')
parser.add_argument('--data_dir', type=str, default='/home/jovyan/gaave/data', help='data directory')
parser.add_argument('--num_classes', type=int, default=7, help='number of emotion classes')
# scheduling hparams
parser.add_argument('--lr_decay_step', type=int, default=10, help='learning rate decay step')
parser.add_argument('--lr_decay_gamma', type=float, default=0.2, help='learning rate decay gamma')

# TCN params
parser.add_argument('--sequential', type=bool, default=False, help='use tcn or not')
parser.add_argument('--seq_length', type=int, default=64, help='sequence length for tcn')

# GAAVE hyperparameters
parser.add_argument('--use_gaave', type=bool, default=True, help="use gaave or not")

parser.add_argument('--head_train_epochs', type=int, default=25,
                    help='Number of training epochs for head model')
parser.add_argument('--tail_train_epochs', type=int, default=25,
                    help='Number of training epochs for tail model')
parser.add_argument('--head_learning_rate', type=float, default=5e-4,
                    help='Learning rate for head model')
parser.add_argument('--tail_learning_rate', type=float, default=5e-4,
                    help='Learning rate for tail model')
parser.add_argument('--attack_learning_rate', type=float, default=5e-4,
                    help='Learning rate for adversarial attack')
parser.add_argument('--attack_head_epochs', type=int, default=1,
                    help='Number of attack epochs for head model')
parser.add_argument('--attack_tail_epochs', type=int, default=1,
                    help='Number of attack epochs for tail model')
# for attacks
parser.add_argument('--max_disturbance_range', type=float, default=1/255,
                    help='Maximum disturbance range (ε value) for PGD attack')
parser.add_argument('--step_size', type=float, default=None,
                    help='Step size (α value) for PGD attack (defaults to max_disturbance_range/8 if None)')
parser.add_argument('--tau_head', type=int, default=1,
                    help='Threshold for identifying noisy samples in head set')
parser.add_argument('--tau_tail', type=int, default=1,
                    help='Threshold for identifying noisy samples in tail set')
# for self annotator
parser.add_argument('--self_annotator_learning_rate', type=float, default=5e-4,
                    help='Learning rate for self annotator model')
parser.add_argument('--self_annotator_epochs', type=int, default=15,
                    help='Number of training epochs for self annotator model')

args = parser.parse_args(args=[])
print(args)

# 3.&nbsp;GAAVE

In [ ]:
# helper function that efficiently calculates the samples per class
def count_samples_per_class(dataset):
    import torch
    from torch.utils.data import DataLoader
    
    loader = DataLoader(
        dataset, 
        batch_size=1024,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )
    
    class_counts = Counter()
    for _, batch_labels in loader:
        if isinstance(batch_labels, torch.Tensor):
            batch_labels = batch_labels.cpu().numpy().tolist()
        class_counts.update(batch_labels)
    
    return class_counts

In [ ]:
# training loop for GAAVE models
def train_with_amp(model, dataloader, optimizer, device, criterion, scaler):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        
        if scaler is not None:
            with torch.amp.autocast('cuda'):
                outputs = model(inputs)
                loss = criterion(outputs, targets)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
        # stats
        _, predicted = outputs.max(1)
        correct += predicted.eq(targets).sum().item()
        total += targets.size(0)
        total_loss += loss.item() * inputs.size(0)
        
    acc = 100. * correct / total
    avg_loss = total_loss / total
    return acc, avg_loss

In [ ]:
class GAAVE:
  def __init__(self, model_class, train_dataset, num_classes, device):
    self.model_class = model_class
    self.train_dataset = train_dataset
    self.num_classes = num_classes
    self.device = device
    self.head_to_tail_class_ratio = 0.4
    
    # Pipeline state storage
    self.head_indices = None
    self.tail_indices = None
    self.head_classes = None
    self.tail_classes = None
    self.head_class_counts = None
    self.head_indices_per_class = None
    
    self.head_subset = None
    self.tail_subset = None
    
    self.head_model = None
    self.tail_model = None
    
    self.head_vulnerability_scores = None
    self.tail_vulnerability_scores = None
    
    self.clean_head_subset = None
    self.noisy_head_subset = None
    self.clean_tail_subset = None
    self.noisy_tail_subset = None
    
    self.head_annotator = None
    self.tail_annotator = None
    
    self.relabeled_head_dataset = None
    self.relabeled_tail_dataset = None
    
    self.purified_dataset = None

  def dataset_splitting(self):
    """
    split the dataset into head and tail sets 
    """
    print("dataset splitting")

    # find head and tail classes
    num_head_classes = math.ceil(self.num_classes*self.head_to_tail_class_ratio)

    # log head to tail class ratio as hparam on tensorboard
    writer = SummaryWriter()
    writer.add_hparams({'head_to_tail_class_ratio': self.head_to_tail_class_ratio}, {})
    writer.close()

    data_loader = DataLoader(
        self.train_dataset, 
        batch_size=256,  
        num_workers=4, 
        pin_memory=True 
    )
    
    # collect labels
    labels = []
    for _, batch_labels in data_loader:
        labels.extend(batch_labels.tolist())
    
    class_counts = Counter(labels)
    sorted_classes = [(cls, class_counts[cls]) for cls in
                  sorted(class_counts, key=class_counts.get, reverse=True)]

    self.head_classes = sorted([cls for cls, _ in sorted_classes[:num_head_classes]])
    self.tail_classes = sorted([cls for cls, _ in sorted_classes[num_head_classes:]])

    most_samples_in_tail = sorted_classes[num_head_classes][1] if num_head_classes < len(sorted_classes) else 0
    total_head_samples = sum(class_counts[cls] for cls in self.head_classes)
    head_class_ratios = {cls: class_counts[cls] / total_head_samples for cls in self.head_classes}
    self.head_class_counts = {cls: int(head_class_ratios[cls] * most_samples_in_tail) for cls in self.head_classes}

    self.head_indices = []
    self.head_indices_per_class = {cls: [] for cls in self.head_classes}
    self.tail_indices = []

    # split indices into head and tail sets
    for i, (_, label) in enumerate(self.train_dataset):
        if label in self.head_classes:
            self.head_indices.append(i)
            self.head_indices_per_class[label].append(i)
        else:
            self.tail_indices.append(i)

    print(f"Head classes: {self.head_classes}")
    print(f"Tail classes: {self.tail_classes}")

  def subset_refactoring(self):
    """
    for head_subset:
    - remap original head classes to 0 to len(head_classes)-1
    - map tail class samples to len(head_classes) (as a new class)

    for tail_subset:
    - remap original tail classes to 0 to len(tail_classes)-1
    - map head class samples to len(tail_classes) (as a new class)
    """
    print("refactoring subsets")
    # head to new_head mappings where head classes are 0 to (len(head_classes) - 1) and tail classes are mapped to (len(num_classes))
    new_head = {cls: i for i, cls in enumerate(self.head_classes)}
    new_head.update({cls: len(self.head_classes) for cls in self.tail_classes})
    # tail to new_tail mappings
    new_tail = {cls: i for i, cls in enumerate(self.tail_classes)}
    new_tail.update({cls: len(self.tail_classes) for cls in self.head_classes})

    class RefactoredDataset(Dataset):
      def __init__(self, original_dataset, indices, mapping):
        self.original_dataset = original_dataset
        self.indices = indices
        self.mapping = mapping

      def __len__(self):
        return len(self.indices)

      def __getitem__(self, idx):
        original_idx = self.indices[idx]
        image, label = self.original_dataset[original_idx]
        new_label = self.mapping[label]
        return image, new_label

    # create head subset with all samples
    all_indices = self.head_indices + self.tail_indices
    self.head_subset = RefactoredDataset(self.train_dataset, all_indices, new_head)

    # create tail subset
    tail_subset_indices = self.tail_indices.copy()
    
    for cls, count in self.head_class_counts.items():
        random_indices = torch.randperm(len(self.head_indices_per_class[cls]))[:count].tolist()
        selected_indices = [self.head_indices_per_class[cls][i] for i in random_indices]
        tail_subset_indices.extend(selected_indices)

    self.tail_subset = RefactoredDataset(self.train_dataset, tail_subset_indices, new_tail)

    print(f"Head subset: {len(self.head_subset)} samples, {len(self.head_classes) + 1} classes")
    print(f"Tail subset: {len(self.tail_subset)} samples, {len(self.tail_classes) + 1} classes")

  def train_base_models(self):
    """
    train two base models, one on head subset, one on tail subset
    """
    print("training two base models")
    self.head_model = self.model_class(num_classes=len(self.head_classes) + 1,
                                  pretrained=True,
                                  pretrained_path=args.pretrained_path)
    self.tail_model = self.model_class(num_classes=len(self.tail_classes) + 1,
                                  pretrained=True,
                                  pretrained_path=args.pretrained_path)

    head_optimizer = optim.Adam(self.head_model.parameters(), lr=args.head_learning_rate, weight_decay=args.weight_decay)
    tail_optimizer = optim.Adam(self.tail_model.parameters(), lr=args.tail_learning_rate, weight_decay=args.weight_decay)

    head_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(head_optimizer, T_max=args.head_train_epochs, eta_min=0)
    tail_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(tail_optimizer, T_max=args.tail_train_epochs, eta_min=0)

    head_criterion = nn.CrossEntropyLoss()
    tail_criterion = nn.CrossEntropyLoss()

    self.head_model.to(self.device)
    self.tail_model.to(self.device)

    head_train_loader = DataLoader(
        self.head_subset, 
        batch_size=args.batch_size, 
        shuffle=True,
        num_workers=4,  
        pin_memory=True
    )
    
    tail_train_loader = DataLoader(
        self.tail_subset, 
        batch_size=args.batch_size, 
        shuffle=True,
        num_workers=4, 
        pin_memory=True 
    )

    # mixed precision training
    scaler = torch.amp.GradScaler() if self.device.type == 'cuda' else None

    # train head model
    for epoch in range(args.head_train_epochs):
        train_acc_head, _ = train_with_amp(self.head_model, head_train_loader, head_optimizer, self.device, head_criterion, scaler=scaler)
        head_scheduler.step()
        print(f"Head model epoch {epoch+1}/{args.head_train_epochs} with training accuracy {train_acc_head}")

    # train tail model
    for epoch in range(args.tail_train_epochs):
        train_acc_tail, _ = train_with_amp(self.tail_model, tail_train_loader, tail_optimizer, self.device, tail_criterion, scaler=scaler)
        tail_scheduler.step()
        print(f"Tail model epoch {epoch+1}/{args.tail_train_epochs} with training accuracy {train_acc_tail}")

  def adversarial_attack(self):
    """
    perform adversarial attacks on head and tail subsets to identify vulnerable samples - aligned with Algorithm 1 in the paper
    """
    print("adversarial attack - aligned with Algorithm 1")
    self.head_model.eval()
    self.tail_model.eval()

    # pgd params
    eps = args.max_disturbance_range
    alpha = args.step_size
    max_steps_head = args.attack_head_epochs
    max_steps_tail = args.attack_tail_epochs
    adversarial_lr = float(args.attack_learning_rate)

    # set new learning rates (as in line 11 of Algorithm 1)
    # head_optimizer = optim.Adam(self.head_model.parameters(), lr=adversarial_lr, weight_decay=args.weight_decay)
    # tail_optimizer = optim.Adam(self.tail_model.parameters(), lr=adversarial_lr, weight_decay=args.weight_decay)
    
    head_loader = DataLoader(
        self.head_subset, 
        batch_size=args.batch_size, 
        shuffle=False,
        num_workers=4,  
        pin_memory=True 
    )
    
    tail_loader = DataLoader(
        self.tail_subset, 
        batch_size=args.batch_size, 
        shuffle=False,
        num_workers=4, 
        pin_memory=True
    )

    self.head_vulnerability_scores = {}
    self.tail_vulnerability_scores = {}

    # attack head samples in batches (lines 12-17 of Algorithm 1)
    print("attacking head samples")
    for batch_idx, (images, labels) in enumerate(head_loader):
        images, labels = images.to(self.device), labels.to(self.device)

        # update model parameters once per batch (Line 15 in Algorithm 1)
        # self.head_model.train()
        # outputs = self.head_model(images)
        # loss = F.cross_entropy(outputs, labels)
        # head_optimizer.zero_grad()
        # loss.backward()
        # head_optimizer.step()
        # self.head_model.eval()
        
        # calculate k_i via Eq 2 and Eq 3 (Line 16 in Algorithm 1)
        batch_scores = self._batch_calculate_vulnerability(
            self.head_model, images, labels, eps, alpha, max_steps_head
        )
        
        # record k_i of every sample (Line 17 in Algorithm 1)
        for i, k_i in enumerate(batch_scores):
            idx = batch_idx * args.batch_size + i
            if idx < len(self.head_subset):  
                self.head_vulnerability_scores[idx] = k_i.item()

    # attack tail samples in batches (Line 18 of Algorithm 1 - similar process for tail)
    print("attacking tail samples")
    for batch_idx, (images, labels) in enumerate(tail_loader):
        images, labels = images.to(self.device), labels.to(self.device)
        
        batch_scores = self._batch_calculate_vulnerability(
            self.tail_model, images, labels, eps, alpha, max_steps_tail
        )
        
        for i, k_i in enumerate(batch_scores):
            idx = batch_idx * args.batch_size + i
            if idx < len(self.tail_subset):
                self.tail_vulnerability_scores[idx] = k_i.item()

    # stats
    head_scores = list(self.head_vulnerability_scores.values())
    tail_scores = list(self.tail_vulnerability_scores.values())

    print(f"Head vulnerability statistics: Mean={np.mean(head_scores):.2f}, Median={np.median(head_scores):.2f}")
    print(f"Tail vulnerability statistics: Mean={np.mean(tail_scores):.2f}, Median={np.median(tail_scores):.2f}")

  def _batch_calculate_vulnerability(self, model, images, labels, eps, alpha, max_steps):
    """    
    Implements Eq 2 and Eq 3 from the paper to calculate vulnerability scores
    
    Args:
        model: The model to attack (model parameters are fixed during attack)
        images: Batch of images (B,C,H,W)
        labels: Ground truth labels (B,)
        eps: Maximum perturbation size
        alpha: Step size for PGD
        max_steps: Maximum number of steps to try
        
    Returns:
        k_i: Tensor of vulnerability scores for each image in the batch
    """
    if alpha is None:
        alpha = eps / 8
        
    batch_size = images.size(0)
    
    # vulnerability scores
    k_i = torch.zeros(batch_size, device=self.device)
    
    # initial predictions
    with torch.no_grad():
        outputs = model(images)
        preds = outputs.max(1)[1]
        
    # mask for samples that are correctly classified initially
    correct_mask = (preds == labels)
    
    # if none are correctly classified initially, return zeros
    if not correct_mask.any():
        return k_i
        
    still_correct = correct_mask.clone()
    
    # ONLY attack samples that were correctly classified
    x_adv = images.clone().detach()
    
    # PGD attack for all steps (this implements Eq 2 and Eq 3)
    for step in range(max_steps):
        # stop attacks if no samples left
        if not still_correct.any():
            break
            
        active_indices = torch.nonzero(still_correct).squeeze(1)

        # clone for gradient calc
        x_active = x_adv[active_indices].clone().detach().requires_grad_(True)
        
        outputs = model(x_active)
        loss = F.cross_entropy(outputs, labels[active_indices])
        
        model.zero_grad()
        loss.backward()
        
        # update adversarial examples (Eq 2)
        with torch.no_grad():
            perturbation = alpha * x_active.grad.sign()
            x_active = x_active + perturbation
            
            # project back to epsilon ball
            original_images = images[active_indices]
            x_active = torch.max(torch.min(x_active, original_images + eps), original_images - eps)
            x_active = torch.clamp(x_active, 0, 1)
            
            x_adv[active_indices] = x_active
            
            # check which are still correctly classified
            outputs = model(x_active)
            new_preds = outputs.max(1)[1]
            still_correct_active = (new_preds == labels[active_indices])
            
            # increment k_i for samples that are still correctly classified (Eq 3)
            k_i[active_indices[still_correct_active]] += 1
            
            # update still_correct mask for next iteration
            still_correct[active_indices] = still_correct_active
    
    return k_i

  def second_dataset_splitting(self):
    """
    split the datasets into clean and noisy subsets based on vulnerability scores.
    Criterion 1
    if k_i < tau_(head/tail) then sample is noisy
    else sample is clean
    """
    print("second dataset splitting")

    head_scores = torch.tensor([self.head_vulnerability_scores[i] for i in range(len(self.head_subset))])
    tail_scores = torch.tensor([self.tail_vulnerability_scores[i] for i in range(len(self.tail_subset))])
    
    head_clean_mask = head_scores >= args.tau_head
    head_noisy_mask = ~head_clean_mask
    
    tail_clean_mask = tail_scores >= args.tau_tail
    tail_noisy_mask = ~tail_clean_mask

    # get indices
    clean_head_indices = torch.nonzero(head_clean_mask).flatten().tolist()
    noisy_head_indices = torch.nonzero(head_noisy_mask).flatten().tolist()
    
    clean_tail_indices = torch.nonzero(tail_clean_mask).flatten().tolist()
    noisy_tail_indices = torch.nonzero(tail_noisy_mask).flatten().tolist()

    self.clean_head_subset = Subset(self.head_subset, clean_head_indices)
    self.noisy_head_subset = Subset(self.head_subset, noisy_head_indices)
    self.clean_tail_subset = Subset(self.tail_subset, clean_tail_indices)
    self.noisy_tail_subset = Subset(self.tail_subset, noisy_tail_indices)

    # stats
    total_head = len(self.head_subset)
    total_tail = len(self.tail_subset)

    print(f"Head subset split: {len(clean_head_indices)} clean samples ({len(clean_head_indices)/total_head:.1%}), "
          f"{len(noisy_head_indices)} noisy samples ({len(noisy_head_indices)/total_head:.1%})")
    print(f"Tail subset split: {len(clean_tail_indices)} clean samples ({len(clean_tail_indices)/total_tail:.1%}), "
          f"{len(noisy_tail_indices)} noisy samples ({len(noisy_tail_indices)/total_tail:.1%})")

  def train_self_annotators(self):
    """
    train self-annotators on clean subsets - one for head, one for tail 
    """
    # create two annotator models
    self.head_annotator = self.model_class(num_classes=len(self.head_classes) + 1,
                                      pretrained=True,
                                      pretrained_path=args.pretrained_path)
    self.tail_annotator = self.model_class(num_classes=len(self.tail_classes) + 1,
                                      pretrained=True,
                                      pretrained_path=args.pretrained_path)

    self.head_annotator.to(self.device)
    self.tail_annotator.to(self.device)

    clean_head_train_loader = DataLoader(
        self.clean_head_subset, 
        batch_size=args.batch_size, 
        shuffle=True,
        num_workers=4, 
        pin_memory=True
    )
    
    clean_tail_train_loader = DataLoader(
        self.clean_tail_subset, 
        batch_size=args.batch_size, 
        shuffle=True,
        num_workers=4, 
        pin_memory=True 
    )

    # optims and criterion
    head_optimizer = optim.Adam(self.head_annotator.parameters(), lr=args.self_annotator_learning_rate, weight_decay=args.weight_decay)
    tail_optimizer = optim.Adam(self.tail_annotator.parameters(), lr=args.self_annotator_learning_rate, weight_decay=args.weight_decay)
    head_criterion = nn.CrossEntropyLoss()
    tail_criterion = nn.CrossEntropyLoss()

    scaler = torch.amp.GradScaler() if self.device.type == 'cuda' else None

    # train head annotator
    for epoch in range(args.self_annotator_epochs):
        train_acc_head_annotator, train_loss_head_annotator = train_with_amp(
            self.head_annotator, clean_head_train_loader, head_optimizer, self.device, head_criterion, scaler=scaler
        )
        print(f"Head annotator epoch {epoch+1}/{args.self_annotator_epochs} with accuracy {train_acc_head_annotator}")

    # train tail annotator
    for epoch in range(args.self_annotator_epochs):
        train_acc_tail_annotator, train_loss_tail_annotator = train_with_amp(
            self.tail_annotator, clean_tail_train_loader, tail_optimizer, self.device, tail_criterion, scaler=scaler
        )
        print(f"Tail annotator epoch {epoch+1}/{args.self_annotator_epochs} with accuracy {train_acc_tail_annotator}")

  def relabel_noisy_samples(self):
    """
    relabel noisy samples using self-annotators - Equation 5 in the paper
    """
    print("Relabeling noisy samples using self-annotators (batch processing)...")
    self.head_annotator.eval()
    self.tail_annotator.eval()

    batch_size = 256
    
    noisy_head_loader = DataLoader(
        self.noisy_head_subset, 
        batch_size=batch_size, 
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )
    
    noisy_tail_loader = DataLoader(
        self.noisy_tail_subset, 
        batch_size=batch_size, 
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )

    # track relabeled samples
    relabeled_head_indices = []
    relabeled_head_new_labels = []
    relabeled_tail_indices = []
    relabeled_tail_new_labels = []

    # track discarded sample count
    head_discarded, tail_discarded = 0, 0

    # relabel noisy head samples in batches
    print("Relabeling head samples...")
    for batch_idx, (images, _) in enumerate(noisy_head_loader):
        images = images.to(self.device)
        batch_size_actual = images.size(0)  

        # get preds from both annotator models in a single batch
        with torch.no_grad():
            head_preds = self.head_annotator(images).max(1)[1]
            tail_preds = self.tail_annotator(images).max(1)[1]
        
        # apply equation 5 using tensor operations
        valid_mask = (head_preds != len(self.head_classes)) & (tail_preds == len(self.tail_classes))
        valid_indices = torch.nonzero(valid_mask).squeeze(1)
        
        # calculate global indices and get predictions for valid samples
        for i in valid_indices:
            global_idx = batch_idx * batch_size + i.item()
            if global_idx < len(self.noisy_head_subset): 
                relabeled_head_indices.append(self.noisy_head_subset.indices[global_idx])
                relabeled_head_new_labels.append(head_preds[i].item())
        
        # count discarded samples
        head_discarded += (batch_size_actual - valid_mask.sum().item())

    # relabel noisy tail samples in batches
    print("Relabeling tail samples...")
    for batch_idx, (images, _) in enumerate(noisy_tail_loader):
        images = images.to(self.device)
        batch_size_actual = images.size(0) 

        # get preds from both annotator models in a single batch
        with torch.no_grad():
            head_preds = self.head_annotator(images).max(1)[1]
            tail_preds = self.tail_annotator(images).max(1)[1]
        
        # eq. 5
        valid_mask = (head_preds == len(self.head_classes)) & (tail_preds != len(self.tail_classes))
        valid_indices = torch.nonzero(valid_mask).squeeze(1)

        for i in valid_indices:
            global_idx = batch_idx * batch_size + i.item()
            if global_idx < len(self.noisy_tail_subset): 
                relabeled_tail_indices.append(self.noisy_tail_subset.indices[global_idx])
                relabeled_tail_new_labels.append(tail_preds[i].item())

        tail_discarded += (batch_size_actual - valid_mask.sum().item())

    class RelabeledDataset(Dataset):
      def __init__(self, original_dataset, indices, new_labels):
        self.original_dataset = original_dataset
        self.indices = indices
        self.new_labels = new_labels

      def __len__(self):
        return len(self.indices)

      def __getitem__(self, idx):
        original_idx = self.indices[idx]
        image, _ = self.original_dataset[original_idx]
        return image, self.new_labels[idx]

    self.relabeled_head_dataset = RelabeledDataset(self.train_dataset, relabeled_head_indices, relabeled_head_new_labels)
    self.relabeled_tail_dataset = RelabeledDataset(self.train_dataset, relabeled_tail_indices, relabeled_tail_new_labels)

    # stats
    print(f"Head subset: {len(self.relabeled_head_dataset)} samples relabeled, {head_discarded} samples discarded")
    print(f"Tail subset: {len(self.relabeled_tail_dataset)} samples relabeled, {tail_discarded} samples discarded")

  def create_purified_dataset(self):
      """
      create final purified dataset by merging clean and relabeled datasets
      """
      print("creating purified dataset")

      index_to_label = {}

      print("Processing clean head samples...")
      for i in range(len(self.clean_head_subset)):
          _, refactored_label = self.clean_head_subset[i]

          if refactored_label < len(self.head_classes):
              refactored_dataset_idx = self.clean_head_subset.indices[i]
              original_idx = self.clean_head_subset.dataset.indices[refactored_dataset_idx]
              original_label = self.head_classes[refactored_label]
              index_to_label[original_idx] = original_label

      print("Processing clean tail samples...")
      for i in range(len(self.clean_tail_subset)):
          _, refactored_label = self.clean_tail_subset[i]

          if refactored_label < len(self.tail_classes):
              refactored_dataset_idx = self.clean_tail_subset.indices[i]
              original_idx = self.clean_tail_subset.dataset.indices[refactored_dataset_idx]

              original_label = self.tail_classes[refactored_label]

              index_to_label[original_idx] = original_label

      print("Processing relabeled head samples...")
      for i in range(len(self.relabeled_head_dataset)):
          original_idx = self.relabeled_head_dataset.indices[i]
          refactored_label = self.relabeled_head_dataset.new_labels[i]
          original_label = self.head_classes[refactored_label]
          index_to_label[original_idx] = original_label

      print("Processing relabeled tail samples...")
      for i in range(len(self.relabeled_tail_dataset)):
          original_idx = self.relabeled_tail_dataset.indices[i]
          refactored_label = self.relabeled_tail_dataset.new_labels[i]
          original_label = self.tail_classes[refactored_label]
          index_to_label[original_idx] = original_label

      class PurifiedDataset(Dataset):
          def __init__(self, original_dataset, index_to_label):
              self.original_dataset = original_dataset
              self.indices = sorted(list(index_to_label.keys()))
              self.labels = [index_to_label[idx] for idx in self.indices]
              
              self.idx_to_position = {idx: pos for pos, idx in enumerate(self.indices)}

          def __len__(self):
              return len(self.indices)

          def __getitem__(self, idx):
              original_idx = self.indices[idx]
              image, _ = self.original_dataset[original_idx]
              return image, self.labels[idx]

      self.purified_dataset = PurifiedDataset(self.train_dataset, index_to_label)

      class_counts = Counter(self.purified_dataset.labels)

      print(f"Purified dataset created with {len(self.purified_dataset)} samples")
      print("Class distribution:")
      for cls in sorted(class_counts.keys()):
          print(f"  Class {cls}: {class_counts[cls]} samples")

      # original_class_counts = count_samples_per_class(self.train_dataset)
      # print("\nComparison with original dataset:")
      # for cls in sorted(original_class_counts.keys()):
      #     original_count = original_class_counts[cls]
      #     purified_count = class_counts.get(cls, 0)
      #     ratio = purified_count / original_count if original_count > 0 else 0
      #     print(f"  Class {cls}: Original {original_count}, Purified {purified_count} ({ratio:.1%})")

  def run_pipeline(self):
    """
    run the complete GAAVE pipeline:
    1. Dataset splitting and subset refactoring
    2. Train two base models
    3. Adversarial attack to identify noisy samples
    4. Split into clean and noisy subsets
    5. Train self-annotators on clean subsets
    6. Relabel noisy samples
    7. Create final purified dataset

    Returns:
        purified_dataset: final dataset with clean and relabeled samples
    """
    print("Running GAAVE pipeline...")
    
    # start timing the pipeline
    start_time = time.time()

    # 1 dataset splitting
    self.dataset_splitting()
    time_checkpoint1 = time.time()
    print(f"Dataset splitting completed in {time_checkpoint1 - start_time:.2f} seconds")
    
    # 2 subset refactoring
    self.subset_refactoring()
    time_checkpoint2 = time.time()
    print(f"Subset refactoring completed in {time_checkpoint2 - time_checkpoint1:.2f} seconds")
    
    ### --- GAAVE (ALGORITHM 1) --- ###
    # 3 training two base models
    self.train_base_models()
    time_checkpoint3 = time.time()
    print(f"Base model training completed in {time_checkpoint3 - time_checkpoint2:.2f} seconds")
    
    # 4 adversarial attack
    self.adversarial_attack()
    time_checkpoint4 = time.time()
    print(f"Adversarial attack completed in {time_checkpoint4 - time_checkpoint3:.2f} seconds")
    
    # 5 splitting dataset a second time
    self.second_dataset_splitting()
    time_checkpoint5 = time.time()
    print(f"Second dataset splitting completed in {time_checkpoint5 - time_checkpoint4:.2f} seconds")
    
    ### --- END OF GAAVE --- ###
    # 6 self annotation
    self.train_self_annotators()
    time_checkpoint6 = time.time()
    print(f"Self-annotator training completed in {time_checkpoint6 - time_checkpoint5:.2f} seconds")
    
    # 7 relabel
    self.relabel_noisy_samples()
    time_checkpoint7 = time.time()
    print(f"Relabeling completed in {time_checkpoint7 - time_checkpoint6:.2f} seconds")
    
    # 8 create final purified dataset
    self.create_purified_dataset()
    time_checkpoint8 = time.time()
    print(f"Purified dataset creation completed in {time_checkpoint8 - time_checkpoint7:.2f} seconds")

    # report total time
    total_time = time_checkpoint8 - start_time
    print(f"GAAVE pipeline completed successfully in {total_time:.2f} seconds ({total_time/60:.2f} minutes)!")
    
    # clean up CUDA cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
    return self.purified_dataset

# 4.&nbsp; Dataset Loader and Main Loop

## Dataset Loaders

In [ ]:
class EmotionDataset(Dataset):
    """
    class that can load datasets from RAF-DB, FER+, AffectNet, affwild2
    """
    def __init__(self, root_dir, dataset_name, split='train', transform=None):
        """
        Args:
            root_dir (string): Directory with all the images.
            dataset_name (string): Name of the dataset (rafdb, affectnet, ferplus)
            split (string): 'train' or 'test'
            transform (callable, optional): Optional transform to be applied on an image
        """

        self.root_dir = root_dir
        self.dataset_name = dataset_name.lower()
        self.split = split
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.video_info = {}

        self._load_dataset()

    def _load_dataset(self):
        if self.dataset_name == 'rafdb':
          self._load_rafdb()
        elif self.dataset_name == 'affectnet7':
          self._load_affectnet7()
        elif self.dataset_name == 'ferplus':
          self._load_ferplus()
        elif self.dataset_name == 'affwild2':
          self._load_affwild2()
        elif self.dataset_name == 'affwild2_sequential':
          self._load_affwild2_sequential()
        else:
          raise ValueError(f"dataset {self.dataset_name} not supported")

    def _load_rafdb(self):
        """
        Load RAF-DB dataset
        RAF-DB structure:
        - Image directory: aligned
        - Label file: list_partition_label.txt

        Notes:
        - images are named "test_0001_aligned.jpg" in the aligned folder
        - but the label file lists them without the "_aligned" suffix, e.g., "test_0001.jpg"
        """
        data_dir = os.path.join(self.root_dir, 'rafdb')
        img_dir = os.path.join(data_dir, 'aligned')

        # try both common filenames for the label file
        label_file = os.path.join(data_dir, 'list_partition_label.txt')
        if not os.path.exists(label_file):
            alt_label_file = os.path.join(data_dir, 'list_patition_label.txt') # my label file name
            if os.path.exists(alt_label_file):
                label_file = alt_label_file
            else:
                raise FileNotFoundError(f"RAF-DB label file not found at {label_file}")

        print(f"Loading RAF-DB from {label_file}")

        with open(label_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 2:
                    continue

                img_name, label = parts

                # check if the filename matches the expected split
                # image names in label file are like "test_0001.jpg" or "train_0001.jpg"
                prefix = img_name.split('_')[0] 

                if prefix == self.split:
                    # strip .jpg extension if present
                    base_name = img_name.rsplit('.', 1)[0] if '.' in img_name else img_name

                    # try multiple possible naming conventions
                    possible_paths = [
                        os.path.join(img_dir, f"{base_name}_aligned.jpg"),  # test_0001_aligned.jpg
                        os.path.join(img_dir, f"{base_name}_aligned.png"),  # test_0001_aligned.png
                        os.path.join(img_dir, f"{base_name}.jpg"),          # test_0001.jpg
                        os.path.join(img_dir, img_name)                     # Exact name from file
                    ]

                    img_path = None
                    for path in possible_paths:
                        if os.path.exists(path):
                            img_path = path
                            break

                    if img_path:
                        self.image_paths.append(img_path)
                        # RAF-DB labels are 1-7, convert to 0-6
                        self.labels.append(int(label) - 1)

        print(f"Loaded {len(self.image_paths)} images for split: {self.split}")

    def _load_affectnet7(self):
        """
        Load AffectNet7 dataset
        AffectNet7 structure:
        - Image directories: affectnet_train, affectnet_valid
        - Label files: 
            - affectnet_train_according_to_emotiw_1.txt, affectnet_valid_according_to_emotiw_1.txt
            - affectnet_train_test.txt, affectnet_valid_test.txt
        
        Each line in the label file has format:
        /path/to/image.jpg label
        where label is an integer in range 0-6
        """
        data_dir = os.path.join(self.root_dir, 'affectnet7')
        
        if self.split == 'train':
            label_file = os.path.join(data_dir, 'affectnet_train_according_to_emotiw.txt')
            img_dir = os.path.join(data_dir, 'train')
        elif self.split == 'test' or self.split == 'valid':
            # Check first for the test annotation file
            label_file = os.path.join(data_dir, 'affectnet_test_according_to_emotiw.txt')
            img_dir = os.path.join(data_dir, 'validation')
        else:
            raise ValueError(f"split {self.split} not recognized for AffectNet7")
        
        if not os.path.exists(label_file):
            raise FileNotFoundError(f"AffectNet7 label file not found at {label_file}")
        
        print(f"loading AffectNet7 from {label_file}")
        
        with open(label_file, 'r') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                    
                parts = line.split()
                if len(parts) < 2:
                    continue
                    
                # The last part is the label, everything before is the path
                label = int(parts[-1])
                original_path = ' '.join(parts[:-1])
                
                # Extract image filename from the original path
                img_filename = os.path.basename(original_path)
                
                # Construct the path to the actual image file in our directory structure
                img_path = os.path.join(img_dir, img_filename)
                
                if os.path.exists(img_path):
                    self.image_paths.append(img_path)
                    self.labels.append(label)
                else:
                    for ext in ['.jpg', '.jpeg', '.png']:
                        base_name = os.path.splitext(img_filename)[0]
                        alt_path = os.path.join(img_dir, base_name + ext)
                        if os.path.exists(alt_path):
                            self.image_paths.append(alt_path)
                            self.labels.append(label)
                            break
        
        print(f"Loaded {len(self.image_paths)} images for split: {self.split}")

    def _load_ferplus(self):
      """
      Load FER+ dataset
      FER+ structure after using generate_training_data.py:
      - FER2013Train/FER2013Valid/FER2013Test folders: containing images and label.csv

      FER+ has 8 emotion categories:
      0=neutral, 1=happiness, 2=surprise, 3=sadness,
      4=anger, 5=disgust, 6=fear, 7=contempt
      """
      import csv

      data_dir = os.path.join(self.root_dir, 'ferplus')

      args.num_classes = 8

      # set the appropriate folder based on split
      if self.split == 'train':
          split_dir = os.path.join(data_dir, 'FER2013Train')
      elif self.split == 'test':
          split_dir = os.path.join(data_dir, 'FER2013Test')
      else:  # valid
          split_dir = os.path.join(data_dir, 'FER2013Valid')

      anno_file = os.path.join(split_dir, 'label.csv')

      if not os.path.exists(anno_file):
          raise FileNotFoundError(f"FER+ annotation file not found at {anno_file}")

      print(f"Loading FER+ from {anno_file}")

      with open(anno_file, 'r') as f:
          reader = csv.reader(f)
          header = next(reader)

          for row in reader:
              if len(row) < 2:  # at least image, label
                  continue

              img_name = row[0]  # image file name (should be something like 'fer0000001.png')

              # the label.csv already has the majority vote computed thus reading the majority vote
              label = int(row[1])

              img_path = os.path.join(split_dir, img_name)

              if os.path.exists(img_path):
                  self.image_paths.append(img_path)
                  self.labels.append(label)
              else:
                  print(f"Warning: Image {img_path} not found")

    def _load_affwild2(self):
      """
      AffWild2 structure after extraction:
      - Root directory contains multiple video directories with frame images
      - annotations/EXPR/train - Contains training annotations
      - annotations/EXPR/test - Contains validation (test) annotations

      AffWild2 has 7 basic expression categories and other:
      0=Neutral, 1=Anger, 2=Disgust, 3=Fear, 4=Happiness, 5=Sadness, 6=Surprise, 7=Other
      """
      import os
      import glob
      from collections import Counter

      args.num_classes = 8

      data_dir = os.path.join(self.root_dir, 'affwild2')

      if not os.path.exists(data_dir):
          raise FileNotFoundError(f"AffWild2 directory not found at {data_dir}")

      print(f"Loading AffWild2 from {data_dir}")

      # path to annotations
      annotations_dir = os.path.join(data_dir, 'annotations', 'EXPR')
      if not os.path.exists(annotations_dir):
          raise FileNotFoundError(f"AffWild2 annotations not found at {annotations_dir}")

      split_dir = 'train' if self.split == 'train' else 'test'
      annotations_split_dir = os.path.join(annotations_dir, split_dir)

      if not os.path.exists(annotations_split_dir):
          raise FileNotFoundError(f"AffWild2 {split_dir} annotations not found at {annotations_split_dir}")

      print(f"Loading {split_dir} annotations from {annotations_split_dir}")

      # track overall class distribution
      class_distribution = {i: 0 for i in range(8)}  # 8 emotion classes (0-7)

      # dictionary to track videos processed
      video_stats = {}

      # iterate through annotation files
      annotation_files = glob.glob(os.path.join(annotations_split_dir, "*.txt"))
      print(f"Found {len(annotation_files)} annotation files")

      for annotation_file in annotation_files:
          # extract video name from filename (e.g., "video1.txt" -> "video1")
          video_name = os.path.splitext(os.path.basename(annotation_file))[0]
          video_dir = os.path.join(data_dir, video_name)

          if not os.path.exists(video_dir):
              print(f"Warning: Video directory not found for {video_name}")
              continue

          # initialise video stats
          video_stats[video_name] = {
              'total_annotations': 0,
              'valid_frames': 0,
              'class_counts': {i: 0 for i in range(8)}
          }

          # available frames
          frame_files = sorted([f for f in os.listdir(video_dir) if f.endswith('.jpg') or f.endswith('.png')])

          if not frame_files:
              print(f"Warning: No frames found for video {video_name}")
              continue

          # frame_index:filename
          frame_map = {}
          for frame_file in frame_files:
              # extract number
              try:
                  frame_num = int(os.path.splitext(frame_file)[0])
                  frame_map[frame_num] = frame_file
              except ValueError:
                  # try other patterns
                  parts = os.path.splitext(frame_file)[0].split('_')
                  if len(parts) > 1 and parts[-1].isdigit():
                      frame_num = int(parts[-1])
                      frame_map[frame_num] = frame_file

          # read the annotation file for labels
          with open(annotation_file, 'r') as f:
              annotations = [line.strip() for line in f.readlines()]

          video_stats[video_name]['total_annotations'] = len(annotations)

          for i, emotion in enumerate(annotations):
              if not emotion.isdigit() and emotion != '-1':
                  try:
                      emotion = int(float(emotion.strip()))
                  except:
                      continue
              else:
                  emotion = int(emotion)

              if emotion < 0 or emotion > 7:
                  continue

              potential_indices = [i+1, i]

              frame_found = False
              for idx in potential_indices:
                  if idx in frame_map:
                      frame_path = os.path.join(video_dir, frame_map[idx])
                      self.image_paths.append(frame_path)
                      self.labels.append(emotion)
                      self.video_info[frame_path] = video_name

                      # Update statistics
                      video_stats[video_name]['valid_frames'] += 1
                      video_stats[video_name]['class_counts'][emotion] += 1
                      class_distribution[emotion] += 1

                      frame_found = True
                      break

      print(f"loaded {len(self.image_paths)} images for split: {self.split}")
      print(f"class distribution: {class_distribution}")

      # print stats
      videos_with_frames = [(v, s) for v, s in video_stats.items() if s['valid_frames'] > 0]
      print(f"\nFound {len(videos_with_frames)} videos with valid frames")

      if videos_with_frames:
          print("\nVideo-specific statistics (top 5):")
          for video, stats in sorted(videos_with_frames, key=lambda x: x[1]['valid_frames'], reverse=True)[:5]:
              print(f"{video}: {stats['valid_frames']}/{stats['total_annotations']} frames matched")
              print(f"class distribution: {dict(stats['class_counts'])}")

    def __len__(self):
      return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        image = Image.open(img_path).convert('RGB')

        if self.transform:
          image = self.transform(image)

        return image, label

class SequentialEmotionDataset(Dataset):
    def __init__(self, root_dir, dataset_name, split='train', transform=None, seq_length=args.seq_length):
        self.root_dir = root_dir
        self.dataset_name = dataset_name
        self.split = split
        self.transform = transform
        self.seq_length = seq_length

        # for storing sequential data
        self.sequences = []        # list of lists of frame paths
        self.sequence_labels = []  # list of sequence labels
        self.video_names = []      # list of video names for each sequence

        # load the appropriate dataset
        if dataset_name.lower() == 'affwild2':
            self._load_affwild2_sequential()
        else:
            raise ValueError(f"Unknown dataset: {dataset_name}")
    def _load_affwild2_sequential(self):
      """
      Load AffWild2 dataset for sequential processing (for TCNs)
      Creates sequences of frames from each video for temporal analysis

      Returns video sequences instead of individual frames
      """
      import os
      import glob
      from collections import Counter, defaultdict

      args.num_classes = 8 

      data_dir = os.path.join(self.root_dir, 'affwild2')
      if not os.path.exists(data_dir):
          raise FileNotFoundError(f"AffWild2 directory not found at {data_dir}")

      print(f"Loading AffWild2 sequential data from {data_dir}")

      annotations_dir = os.path.join(data_dir, 'annotations', 'EXPR')
      if not os.path.exists(annotations_dir):
          raise FileNotFoundError(f"AffWild2 annotations not found at {annotations_dir}")

      split_dir = 'train' if self.split == 'train' else 'test'
      annotations_split_dir = os.path.join(annotations_dir, split_dir)

      if not os.path.exists(annotations_split_dir):
          raise FileNotFoundError(f"AffWild2 {split_dir} annotations not found at {annotations_split_dir}")

      print(f"Loading {split_dir} annotations from {annotations_split_dir}")

      # dictionary to store sequences for each video
      video_sequences = defaultdict(list)
      video_labels = defaultdict(list)

      # define sequence length (should match model's seq_length)
      self.seq_length = args.seq_length 

      # track overall class distribution
      class_distribution = {i: 0 for i in range(8)}  # 8 emotion classes (0-7)

      # iterate through annotation files
      annotation_files = glob.glob(os.path.join(annotations_split_dir, "*.txt"))
      print(f"Found {len(annotation_files)} annotation files")

      for annotation_file in annotation_files:
          # extract video name from filename
          video_name = os.path.splitext(os.path.basename(annotation_file))[0]
          video_dir = os.path.join(data_dir, video_name)

          if not os.path.exists(video_dir):
              print(f"Warning: Video directory not found for {video_name}")
              continue

          # list all available frames for video
          frame_files = sorted([f for f in os.listdir(video_dir) if f.endswith('.jpg') or f.endswith('.png')])

          if not frame_files:
              print(f"Warning: No frames found for video {video_name}")
              continue

          # idx:filename
          frame_map = {}
          for frame_file in frame_files:
              # frame number
              try:
                  frame_num = int(os.path.splitext(frame_file)[0])
                  frame_map[frame_num] = frame_file
              except ValueError:
                  parts = os.path.splitext(frame_file)[0].split('_')
                  if len(parts) > 1 and parts[-1].isdigit():
                      frame_num = int(parts[-1])
                      frame_map[frame_num] = frame_file

          # read annotation file to get labels
          with open(annotation_file, 'r') as f:
              annotations = [line.strip() for line in f.readlines()]

          # track valid frames and their paths+labels for this video
          valid_frames = []
          valid_labels = []

          # iterate through annotations and find corresponding frames
          for i, emotion in enumerate(annotations):
              # skip invalid annotations
              if not emotion.isdigit() and emotion != '-1':
                  try:
                      emotion = int(float(emotion.strip()))
                  except:
                      continue
              else:
                  emotion = int(emotion)

              if emotion < 0 or emotion > 7:
                  continue

              potential_indices = [i+1, i] 

              for idx in potential_indices:
                  if idx in frame_map:
                      frame_path = os.path.join(video_dir, frame_map[idx])
                      valid_frames.append(frame_path)
                      valid_labels.append(emotion)
                      class_distribution[emotion] += 1
                      break

          # now create sequences of consecutive frames
          if len(valid_frames) >= self.seq_length:
              # create overlapping sequences if desired
              stride = self.seq_length // 2  # 50% overlap

              for i in range(0, len(valid_frames) - self.seq_length + 1, stride):
                  sequence_frames = valid_frames[i:i + self.seq_length]
                  sequence_labels = valid_labels[i:i + self.seq_length]

                  # use the most frequent label as the sequence label
                  label_counts = Counter(sequence_labels)
                  most_common_label = label_counts.most_common(1)[0][0]

                  self.sequences.append(sequence_frames)
                  self.sequence_labels.append(most_common_label)
                  self.video_names.append(video_name)

      print(f"Loaded {len(self.sequences)} sequences for split: {self.split}")
      print(f"Class distribution: {class_distribution}")
      print(f"Average sequence length: {self.seq_length}")

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        sequence_paths = self.sequences[idx]
        label = self.sequence_labels[idx]

        # load all frames in the sequence
        sequence_frames = []
        for frame_path in sequence_paths:
            img = Image.open(frame_path).convert('RGB')
            if self.transform:
                img = self.transform(img)
            sequence_frames.append(img)

        # frames of shape [seq_length, channels, height, width]
        sequence_tensor = torch.stack(sequence_frames)

        return sequence_tensor, label

In [ ]:
def apply_gaave_to_sequential_data(train_transform):
    """
    apply GAAVE to AffWild2 dataset while preserving temporality for TCN training
    """
    print("applying GAAVE to AffWild2 for temporal data")
    
    # frame level dataset for GAAVE
    frame_dataset = EmotionDataset(
        root_dir=args.data_dir,
        dataset_name='affwild2',
        split='train',
        transform=train_transform
    )
    
    # rim GAAVE
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    gaave = GAAVE(ResNet18Emotion, frame_dataset, args.num_classes, device)
    purified_frame_dataset = gaave.run_pipeline()
    
    # see which frames are in the purified dataset
    purified_indices_set = set(purified_frame_dataset.indices)
    
    # og index: purified label
    idx_to_purified_label = {}
    for i, idx in enumerate(purified_frame_dataset.indices):
        idx_to_purified_label[idx] = purified_frame_dataset.labels[i]
    
    # group frames for sequences
    video_to_frames = {}
    
    for i, img_path in enumerate(frame_dataset.image_paths):
        # skip deleted frames
        if i not in purified_indices_set:
            continue
            
        video_name = frame_dataset.video_info.get(img_path)
        if not video_name:
            video_name = os.path.basename(os.path.dirname(img_path))
        
        if video_name not in video_to_frames:
            video_to_frames[video_name] = []
        
        frame_path = img_path
        frame_num = extract_frame_number(frame_path)
        purified_label = idx_to_purified_label[i]
        
        video_to_frames[video_name].append((frame_path, frame_num, purified_label))
    
    # create sequences
    sequences = []
    sequence_labels = []
    video_names = []
    
    for video_name, frames in video_to_frames.items():
        # Sort frames by their position in the video
        frames.sort(key=lambda x: x[1])
        
        # Create sequences with stride for overlap
        if len(frames) >= args.seq_length:
            stride = args.seq_length // 2  # 50% overlap
            
            for i in range(0, len(frames) - args.seq_length + 1, stride):
                seq_frames = frames[i:i + args.seq_length]
                frame_paths = [f[0] for f in seq_frames]
                frame_labels = [f[2] for f in seq_frames]
                
                # Use majority voting for sequence label
                label_counts = Counter(frame_labels)
                most_common_label = label_counts.most_common(1)[0][0]
                
                sequences.append(frame_paths)
                sequence_labels.append(most_common_label)
                video_names.append(video_name)
    
    print(f"Created {len(sequences)} sequences from purified frames")
    
    # create custom sequence dataset (same as in datasetloaders)
    class PurifiedSequentialDataset(Dataset):
        def __init__(self, sequences, labels, video_names, transform):
            self.sequences = sequences
            self.sequence_labels = labels
            self.video_names = video_names
            self.transform = transform
            self.seq_length = args.seq_length
            
        def __len__(self):
            return len(self.sequences)
            
        def __getitem__(self, idx):
            sequence_paths = self.sequences[idx]
            label = self.sequence_labels[idx]
            
            sequence_frames = []
            for frame_path in sequence_paths:
                img = Image.open(frame_path).convert('RGB')
                if self.transform:
                    img = self.transform(img)
                sequence_frames.append(img)
                
            # [seq_length, channels, height, width]
            sequence_tensor = torch.stack(sequence_frames)
            return sequence_tensor, label
    
    purified_sequential_dataset = PurifiedSequentialDataset(
        sequences, sequence_labels, video_names, train_transform
    )
    
    print(f"Final purified sequential dataset has {len(purified_sequential_dataset)} sequences")
    return purified_sequential_dataset

def extract_frame_number(path):
    """Extract frame number from image path"""
    filename = os.path.basename(path)
    base_name = os.path.splitext(filename)[0]
    
    try:
        # Simple number (e.g., "00001.jpg")
        return int(base_name)
    except ValueError:
        try:
            parts = base_name.split('_')
            if len(parts) > 1 and parts[-1].isdigit():
                return int(parts[-1])
        except:
            return base_name

## Model and Main Loop

In [ ]:
class ResNet18Emotion(nn.Module):
    def __init__(self, num_classes=7, pretrained=True, pretrained_path=args.pretrained_path):
        super(ResNet18Emotion, self).__init__()
        self.model = resnet18(weights=None)
        if pretrained:
            if pretrained_path:
                print(f"Loading pretrained model from {pretrained_path}")
                try:
                    device = next(self.model.parameters()).device
                    checkpoint = torch.load(pretrained_path, map_location=device)

                    # different checkpoint formats
                    if isinstance(checkpoint, dict):
                        if 'state_dict' in checkpoint:
                            print("Found nested state_dict in checkpoint")
                            state_dict = {k.replace('module.', ''): v for k, v in checkpoint['state_dict'].items()}
                        else:
                            print("Using checkpoint as direct state_dict")
                            state_dict = checkpoint
                    else:
                        print("Checkpoint is not a dictionary, using as is")
                        state_dict = checkpoint

                    # load state_dict
                    missing_keys, unexpected_keys = self.model.load_state_dict(state_dict, strict=False)

                    print(f"Successfully loaded weights with {len(missing_keys)} missing and {len(unexpected_keys)} unexpected keys")
                    if missing_keys:
                        print(f"Missing keys: {missing_keys[:5]}{'...' if len(missing_keys) > 5 else ''}")
                    if unexpected_keys:
                        print(f"Unexpected keys: {unexpected_keys[:5]}{'...' if len(unexpected_keys) > 5 else ''}")

                except Exception as e:
                    print(f"Error loading weights: {e}")
                    print("Falling back to ImageNet weights")
                    self.model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
                    print("Loaded ResNet18 with pretrained IMAGENET1K weights")
            else:
                self.model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
                print("Loaded ResNet18 with pretrained IMAGENET1K weights")
        else:
            self.model = resnet18(weights=None)
            print("Loaded ResNet18 without pretrained weights")

        num_features = self.model.fc.in_features
        self.model.fc = nn.Sequential(
            nn.Linear(num_features, num_classes)
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
class Chomp1d(nn.Module):
    """
    remove padding at the end of the sequence to maintain temporal causality
    """
    def __init__(self, chomp_size):
        super(Chomp1d, self).__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous() if self.chomp_size > 0 else x

In [ ]:
# code inspired by https://github.com/flrngel/TCN-with-attention/blob/master/tcn.py
# relu is used to stay consistent with resnet18
# although layernorm is used to better accomodate TCNs
class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding):
        super(TemporalBlock, self).__init__()

        self.conv1 = nn.utils.parametrizations.weight_norm(nn.Conv1d(
            n_inputs, n_outputs, kernel_size,
            stride=stride, padding=padding, dilation=dilation))

        self.chomp1 = Chomp1d(padding)
        # self.bn1 = nn.BatchNorm1d(n_outputs)
        self.ln1 = nn.GroupNorm(num_groups=32, num_channels=n_outputs)
        self.relu1 = nn.ReLU()

        self.conv2 = nn.utils.parametrizations.weight_norm(nn.Conv1d(
            n_outputs, n_outputs, kernel_size,
            stride=stride, padding=padding, dilation=dilation))

        self.chomp2 = Chomp1d(padding)
        self.ln2 = nn.GroupNorm(num_groups=32, num_channels=n_outputs)
        self.relu2 = nn.ReLU()

        self.net = nn.Sequential(
            self.conv1, self.chomp1, self.ln1, self.relu1,
            nn.Dropout(0.2),
            self.conv2, self.chomp2, self.ln2, self.relu2,
            nn.Dropout(0.2)
        )

        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None

        self.relu = nn.ReLU()

        self.init_weights()

    def init_weights(self):
        nn.init.kaiming_normal_(self.conv1.weight.data, mode='fan_in', nonlinearity='relu')
        nn.init.kaiming_normal_(self.conv2.weight.data, mode='fan_in', nonlinearity='relu')
        if self.downsample is not None:
            nn.init.kaiming_normal_(self.downsample.weight.data, mode='fan_in', nonlinearity='relu')

    def forward(self, x):
        # Check for NaN inputs (debugging)
        if torch.isnan(x).any():
            print("NaN detected in TemporalBlock input")
            # x = torch.nan_to_num(x, nan=0.0)

        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        
        if out.size(2) != res.size(2):
            # Adjust the longer one to match the shorter one
            min_len = min(out.size(2), res.size(2))
            out = out[:, :, :min_len]
            res = res[:, :, :min_len]

        return self.relu(out + res)

# for layers
class TemporalConvNet(nn.Module):
  def __init__(self, num_inputs, num_channels, kernel_size=2):
    super(TemporalConvNet, self).__init__()

    layers = []
    num_levels = len(num_channels)

    for i in range(num_levels):
      dilation_size = 2 ** i
      in_channels = num_inputs if i == 0 else num_channels[i-1]
      out_channels = num_channels[i]
      padding = (kernel_size - 1) * dilation_size

      layers.append(
        TemporalBlock(
            in_channels, out_channels, kernel_size, stride=1,
            dilation=dilation_size, padding=padding)
      )

    self.network = nn.Sequential(*layers)

  def forward(self, x):
    return self.network(x)

In [ ]:
class ResNet18TCN(nn.Module):
  def __init__(self, num_classes=8, pretrained=True, seq_length=args.seq_length):
    super(ResNet18TCN, self).__init__()

    # extract spatial features (resnet18 w/o the final FC layer)
    resnet = ResNet18Emotion(num_classes=num_classes, pretrained=True, pretrained_path=args.pretrained_path)
    self.feature_extractor = nn.Sequential(*list(resnet.model.children())[:-1])
    feature_dim = 512 #resnet18 output dims

    #TCN params
    self.seq_length = seq_length
    kernel_size = 3

    # get the amount of layers needed for full coverage through dilation
    # log2 because we will be increasing dilation by 2^i every layer
    num_layers = math.ceil(math.log2((seq_length-1) / (kernel_size - 1) + 1))
    print('num_layers for TCN: ', num_layers)

    # TCN itself
    num_channels = [64, 128, 256, 512][:num_layers]
    if len(num_channels) < num_layers:
        num_channels.extend([512] * (num_layers - len(num_channels)))
    self.tcn = TemporalConvNet(
        num_inputs=feature_dim,
        num_channels=num_channels,
        kernel_size=kernel_size
    )

    self.classifier = nn.Linear(num_channels[-1], num_classes)

    self._initialise_weights()

  def _initialise_weights(self):
    nn.init.kaiming_normal_(self.classifier.weight, mode='fan_out', nonlinearity='relu')
    nn.init.constant_(self.classifier.bias, 0)


  def forward(self, x):
    if torch.isnan(x).any():
      print("NaN detected in model input")
        
    batch_size, seq_length = x.size(0), x.size(1)
    # [batch size * seq_length, channels, height, width]
    x = x.view(-1, x.size(2), x.size(3), x.size(4))
    # [batch_size * seq_length, 512, 1, 1]
    features = self.feature_extractor(x)
    # remove spatial dimensions [bs * sl, 512]
    features = features.squeeze(-1).squeeze(-1)  

    # back to seq format [bs, seq_length, features]
    features = features.view(batch_size, seq_length, -1)

    # for TCN [batch, channels, seq_len]
    features = features.permute(0, 2, 1)

    temporal_features = self.tcn(features)

    # global avg pooling [bs, final chans] 
    pooled_features = torch.mean(temporal_features, dim=2)

    # classification
    output = self.classifier(pooled_features)

    return output

In [ ]:
def train_standard(model, train_loader, optimizer, device, criterion):
    model.train()
    running_loss = 0
    correct = 0
    
    total = 0

    for i, data in enumerate(train_loader):
        inputs, labels = data
        inputs = inputs.to(device)
        labels = labels.to(device)

        # clear grads
        optimizer.zero_grad()

        # forward
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # backwards pass
        loss.backward()
        optimizer.step()

        # stats
        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    return train_acc, train_loss


def train_sequential(model, train_loader, optimizer, device, criterion):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    # Track batches with NaN issues for reporting
    nan_detected_batches = 0
    total_batches = len(train_loader)

    for i, data in enumerate(train_loader):
        inputs, labels = data
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Clear gradients
        optimizer.zero_grad()

        try:
            # Forward pass
            outputs = model(inputs)

            # Check for NaN in outputs
            if torch.isnan(outputs).any():
                nan_detected_batches += 1
                print(f"Warning: NaN detected in outputs for batch {i}/{total_batches}")
                continue

            # Calculate loss
            loss = criterion(outputs, labels)

            # Check for NaN in loss
            if torch.isnan(loss).any():
                nan_detected_batches += 1
                print(f"Warning: NaN detected in loss for batch {i}/{total_batches}")
                continue

            # Backward pass
            loss.backward()

            # Clip gradients to prevent explosion
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            # Update weights
            optimizer.step()

            # Update statistics
            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

        except RuntimeError as e:
            print(f"RuntimeError in batch {i}/{total_batches}: {str(e)}")
            continue

    # Report NaN statistics
    if nan_detected_batches > 0:
        print(f"NaN detected in {nan_detected_batches}/{total_batches} batches ({nan_detected_batches/total_batches*100:.2f}%)")

    # Avoid division by zero
    train_loss = running_loss / max(1, len(train_loader) - nan_detected_batches)
    train_acc = correct / max(1, total)

    return train_acc, train_loss


def test_standard(model, test_loader, device, criterion):
    model.eval()
    running_loss = 0
    correct = 0
    total = 0

    # for mean class acc. init a confusion matrix
    num_classes = args.num_classes
    confusion_matrix = torch.zeros(num_classes, num_classes)

    with torch.no_grad():
      for i, data in enumerate(test_loader):
        inputs, labels = data
        inputs = inputs.to(device)
        labels = labels.to(device)

        # forward
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # stats
        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

        # update confusion matrix
        for t, p in zip(labels.view(-1), preds.view(-1)):
            confusion_matrix[t.long(), p.long()] += 1

    # per-class acc.
    per_class_acc = confusion_matrix.diag() / confusion_matrix.sum(1)
    per_class_acc = torch.nan_to_num(per_class_acc, nan=0.0)

    # mean class acc.
    mean_class_acc = per_class_acc.mean().item()

    test_loss = running_loss / len(test_loader)
    test_acc = correct / total

    return test_acc, test_loss, per_class_acc, mean_class_acc


def test_sequential(model, test_loader, device, criterion):
    model.eval()
    running_loss = 0
    correct = 0
    total = 0

    num_classes = model.classifier.out_features 
    confusion_matrix = torch.zeros(num_classes, num_classes)

    with torch.no_grad():
        for i, data in enumerate(test_loader):
            inputs, labels = data
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(inputs)

            if torch.isnan(outputs).any():
                print(f"Warning: NaN detected in test batch {i}, skipping")
                continue

            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

            # Update confusion matrix
            for t, p in zip(labels.view(-1), preds.view(-1)):
                confusion_matrix[t.long(), p.long()] += 1

    per_class_acc = torch.zeros(num_classes)
    for i in range(num_classes):
        if confusion_matrix[i].sum().item() > 0:
            per_class_acc[i] = confusion_matrix[i, i] / confusion_matrix[i].sum()

    mean_class_acc = per_class_acc.mean().item()

    test_loss = running_loss / max(1, len(test_loader))
    test_acc = correct / max(1, total)

    return test_acc, test_loss, per_class_acc, mean_class_acc

In [ ]:
def main():
    try:
        # torch.manual_seed(0)
        # np.random.seed(0)
        print(f"CUDA available: {torch.cuda.is_available()}")
        if torch.cuda.is_available():
            print(f"CUDA device name: {torch.cuda.get_device_name(0)}")

        # Create directories
        os.makedirs('runs', exist_ok=True)
        os.makedirs('checkpoints', exist_ok=True)

        # Create log directory for TensorBoard
        current_time = datetime.now().strftime('%b%d_%H-%M-%S')
        log_dir = os.path.join(
            'runs',
            f'{current_time}_{args.dataset}_lr{args.learning_rate}_bs{args.batch_size}_opt{args.optimizer}'
        )
        print(f"Log directory: {log_dir}")
        writer = SummaryWriter(log_dir)

        # data augmentation and normalisation
        # Note: each dataset have different normalisations - might have to change for each
        train_transform = transforms.Compose([
          transforms.Resize((224, 224)),
          transforms.ToTensor()
          # transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]) # too many samples misclassified with normalisations
        ])

        test_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor()
            # transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])

        if args.sequential:
          args.learning_rate = args.learning_rate * 0.05 # 5e-5
          args.batch_size = 16
          # create datasets
          print(f"Loading training dataset: {args.dataset} from {args.data_dir}")
          train_data = SequentialEmotionDataset(
              root_dir=args.data_dir,
              dataset_name=args.dataset,
              split='train',
              transform=train_transform,
              seq_length=args.seq_length
          )
          print(f"Training dataset loaded successfully with {len(train_data)} sequences")

          print(f"Loading test dataset: {args.dataset}")
          test_data = SequentialEmotionDataset(
              root_dir=args.data_dir,
              dataset_name=args.dataset,
              split='test',
              transform=test_transform,
              seq_length=args.seq_length
          )
          print(f"Test dataset loaded successfully with {len(test_data)} sequences")
        else:
          # create datasets
          print(f"Loading training dataset: {args.dataset} from {args.data_dir}")
          train_data = EmotionDataset(
              root_dir=args.data_dir,
              dataset_name=args.dataset,
              split='train',
              transform=train_transform
          )
          print(f"Training dataset loaded successfully with {len(train_data)} samples")

          print(f"Loading test dataset: {args.dataset}")
          test_data = EmotionDataset(
              root_dir=args.data_dir,
              dataset_name=args.dataset,
              split='test',
              transform=test_transform
          )
          print(f"Test dataset loaded successfully with {len(test_data)} samples")

        # select device (GPU if available)
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {device}")

        train_dataset = None
        if args.use_gaave:
          # GAAVE STUFF
          if args.sequential:
              test_dataset = test_gaave_sequential_approach(train_transform)
              print("Test successful!")
              train_dataset = apply_gaave_to_sequential_data(train_transform)
          else:
              gaave = GAAVE(ResNet18Emotion, train_data, args.num_classes, device)
              purified_dataset = gaave.run_pipeline()
    
              train_dataset = purified_dataset

        else:
          train_dataset = train_data


        train_loader = None
        if args.dataset.lower() == 'affectnet7':
            print("using oversampling for AffectNet7")
            
            # calc class weights for balancing
            labels = np.array(train_dataset.labels)
            class_counts = np.bincount(labels)
            max_count = max(class_counts)
            class_weights = max_count / class_counts
            sample_weights = [class_weights[label] for label in labels]
            
            sampler = WeightedRandomSampler(
                weights=sample_weights,
                num_samples=len(labels),
                replacement=True
            )
            
            train_loader = DataLoader(
                train_dataset,
                batch_size=args.batch_size,
                sampler=sampler, 
                pin_memory=True,
                num_workers=2,
                prefetch_factor=2
            )
        else:
            train_loader = DataLoader(
                train_dataset,
                batch_size=args.batch_size,
                shuffle=True,
                pin_memory=True,
                num_workers=2,
                prefetch_factor=2
            )

        test_loader = DataLoader(
            test_data,
            batch_size=args.batch_size,
            shuffle=False,
            pin_memory=True,
            num_workers=2,
            prefetch_factor=2
        )

        if args.sequential:
          model = ResNet18TCN(
              num_classes=args.num_classes,
              pretrained=True,
              seq_length=args.seq_length
          ).to(device)
          print("using resnet18tcn model (with tcn)")

        else:
          model = ResNet18Emotion(
              num_classes=args.num_classes,
              pretrained=True,
              pretrained_path=args.pretrained_path
          ).to(device)
          print("using resnet18emotion model (non-tcn)")

        print(f'model is on CUDA: {next(model.parameters()).is_cuda}')

        criterion = nn.CrossEntropyLoss()

        if args.optimizer.upper() == 'ADAM':
            optimizer = optim.Adam(
                model.parameters(),
                lr=args.learning_rate,
                weight_decay=args.weight_decay
            )
            print('ADAM optimizer selected')
        elif args.optimizer.upper() == 'SGD':
            optimizer = optim.SGD(
                model.parameters(),
                lr=args.learning_rate,
                weight_decay=args.weight_decay
            )
            print('SGD optimizer selected')
        else:
            print(f"Warning: Optimizer {args.optimizer} not recognized, defaulting to ADAM")
            optimizer = optim.Adam(
                model.parameters(),
                lr=args.learning_rate,
                weight_decay=args.weight_decay
            )

        # tensorboard - log hyperparameters
        try:
            hparams = {
                'learning_rate': args.learning_rate,
                'batch_size': args.batch_size,
                'epochs': args.epochs,
                'optimizer': args.optimizer,
                'weight_decay': args.weight_decay,
                # 'momentum': args.momentum,
                'dataset': args.dataset,
                'num_classes': args.num_classes
            }
            metric_dict = {'hparam/accuracy': 0}
            writer.add_hparams(hparams, metric_dict)
        except Exception as e:
            print(f"Warning: Error logging hyperparameters: {e}")

        # for training loop
        best_acc = 0
        # for avg acc. over last 10 epochs
        test_acc_ten_epochs = []

        for epoch in range(1, args.epochs + 1):
            start_time = time.time()                

            # train and get metrics
            if args.sequential:
              train_acc, train_loss = train_sequential(model, train_loader, optimizer, device, criterion)
              test_acc, test_loss, per_class_acc, mean_class_acc  = test_sequential(model, test_loader, device, criterion)
            else:
              train_acc, train_loss = train_standard(model, train_loader, optimizer, device, criterion)
              test_acc, test_loss, per_class_acc, mean_class_acc  = test_standard(model, test_loader, device, criterion)

            test_acc_ten_epochs.append(test_acc)
            if len(test_acc_ten_epochs) > 10:
                test_acc_ten_epochs.pop(0)

            # logging metrics
            writer.add_scalar('Loss/train', train_loss, epoch)
            writer.add_scalar('Loss/test', test_loss, epoch)
            writer.add_scalar('Accuracy/train', train_acc, epoch)
            writer.add_scalar('Accuracy/test', test_acc, epoch)
            writer.add_scalar('Accuracy/mean_class', mean_class_acc, epoch)
            writer.add_scalar('Accuracy/ten_epochs_mean', np.mean(test_acc_ten_epochs), epoch)
            for i in range(args.num_classes):
                writer.add_scalar(f'Accuracy/class_{i}', per_class_acc[i].item(), epoch)

            # saving best model
            if test_acc > best_acc:
                best_acc = test_acc
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'train_loss': train_loss,
                    'test_loss': test_loss,
                    'train_acc': train_acc,
                    'test_acc': test_acc,
                }, f'./checkpoints/best_model_{args.dataset}_epoch{epoch}.pth')
                print(f"Saved best model with accuracy {best_acc:.4f}")

            # epoch summary
            time_elapsed = time.time() - start_time
            print(f'Epoch {epoch}/{args.epochs} runtime: {time_elapsed:.2f} seconds')
            print(f'Train Acc: {train_acc:.4f}, Train Loss: {train_loss:.4f}')
            print(f'Test Acc: {test_acc:.4f}, Test Loss: {test_loss:.4f}')
            print(f'Mean Class Acc: {mean_class_acc:.4f}')
            print(f'Per-Class Acc: {per_class_acc}')
            print(f'Ten Epochs Mean Acc: {np.mean(test_acc_ten_epochs):.4f}')
            print('-' * 60)

        writer.close()
        print(f'Training complete! Logs saved to {log_dir}')
        print(f'Best test accuracy: {best_acc:.4f}')
        print()

    except Exception as e:
        print(f"An error occurred during execution: {e}")
        import traceback
        traceback.print_exc()

if __name__ == '__main__':
    main()